In [172]:
import cv2
import numpy as np

In [173]:
imagen = cv2.imread("img/bailarina.png", cv2.IMREAD_GRAYSCALE)
_,imagen_bin = cv2.threshold(imagen, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
cv2.imshow("Imagen", imagen_bin)
cv2.waitKey(0)

-1

In [174]:
# Matriz de prueba
"""
imagen_bin = np.matrix([
    [1,1,1],
    [1,0,1],
    [1,1,1]
]).astype(np.uint8)
imagen_bin
"""

'\nimagen_bin = np.matrix([\n    [1,1,1],\n    [1,0,1],\n    [1,1,1]\n]).astype(np.uint8)\nimagen_bin\n'

In [175]:
h, w = imagen_bin.shape
matriz_distancias = (h + w) * imagen_bin.astype(np.uint16)
matriz_distancias

array([[62984, 62984, 62984, ..., 62984, 62984, 62984],
       [62984, 62984, 62984, ..., 62984, 62984, 62984],
       [62984, 62984, 62984, ..., 62984, 62984, 62984],
       ...,
       [62984, 62984, 62984, ..., 62984, 62984, 62984],
       [62984, 62984, 62984, ..., 62984, 62984, 62984],
       [62984, 62984, 62984, ..., 62984, 62984, 62984]],
      shape=(280, 224), dtype=uint16)

In [176]:
# Primera pasada
for y in range(h):
    for x in range(w):
        norte = y - 1
        oeste = x - 1
        if norte < h and norte >= 0:
            pixel_norte = matriz_distancias[norte, x]
        else:
            pixel_norte = h + w
        if oeste < w and oeste >= 0:
            pixel_oeste = matriz_distancias[y, oeste]
        else:
            pixel_oeste = h + w
    
        matriz_distancias[y,x] = min(matriz_distancias[y,x], pixel_norte + 1, pixel_oeste + 1)

print(matriz_distancias)

[[505 505 505 ... 505 505 505]
 [505 506 506 ... 506 506 506]
 [505 506 507 ... 507 507 507]
 ...
 [505 506 507 ... 117 118 119]
 [505 506 507 ... 118 119 120]
 [505 506 507 ... 119 120 121]]


In [177]:
# Segunda Pasada
for y in range(h-1, -1, -1):
    for x in range(w-1, -1, -1):
        sur = y + 1
        este = x + 1
        if sur < h and sur >= 0:
            pixel_sur = matriz_distancias[sur, x]
        else:
            pixel_sur = h + w
        if este < w and este >= 0:
            pixel_este = matriz_distancias[y, este]
        else:
            pixel_este = h + w
        
        matriz_distancias[y,x] = min(matriz_distancias[y,x], pixel_sur + 1, pixel_este + 1)

print(matriz_distancias)

[[ 94  93  92 ... 148 149 150]
 [ 93  92  91 ... 147 148 149]
 [ 92  91  90 ... 146 147 148]
 ...
 [ 93  92  91 ... 117 118 119]
 [ 94  93  92 ... 118 119 120]
 [ 95  94  93 ... 119 120 121]]


In [178]:
matriz_distancias

array([[ 94,  93,  92, ..., 148, 149, 150],
       [ 93,  92,  91, ..., 147, 148, 149],
       [ 92,  91,  90, ..., 146, 147, 148],
       ...,
       [ 93,  92,  91, ..., 117, 118, 119],
       [ 94,  93,  92, ..., 118, 119, 120],
       [ 95,  94,  93, ..., 119, 120, 121]],
      shape=(280, 224), dtype=uint16)

In [ ]:
imagen_bin = cv2.morphologyEx(imagen_bin, cv2.MORPH_GRADIENT, np.ones((3,3)))
imagen_bin = cv2.dilate(imagen_bin, np.ones((3,3)), iterations=4)
cv2.imshow("Process", imagen_bin)
cv2.waitKey(0)
matriz_distancias_2 = cv2.distanceTransform(imagen_bin, cv2.DIST_L1, 3)

In [180]:
(matriz_distancias - matriz_distancias_2).sum()

np.float32(2.430537e+06)

In [181]:
%pip install scikit-image

Note: you may need to restart the kernel to use updated packages.


In [182]:
from skimage.feature import peak_local_max
skeleton = np.zeros(imagen_bin.shape, dtype=np.uint8)
coords = peak_local_max(matriz_distancias_2, min_distance=1)
for coord in coords:
    skeleton[coord[0], coord[1]] = 255
cv2.imshow("Distancias", skeleton)
cv2.waitKey(0)

-1